In [1]:
import sys
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

ROOT = Path(r"d:\ML Project - E-commerce product search with semantic ranking")
DATA = ROOT / "data"
sys.path.append(str(ROOT))

from src.evaluate import evaluate_run, build_run_dict

qrels = pd.read_parquet(DATA / "qrels.parquet")
queries = pd.read_parquet(DATA / "queries.parquet")
bm25_run = pd.read_parquet(DATA / "run_bm25.parquet")
sem_run = pd.read_parquet(DATA / "run_semantic.parquet")

bm_d = build_run_dict(bm25_run)
sm_d = build_run_dict(sem_run)
print("BM25 queries:", len(bm_d), "| Semantic queries:", len(sm_d))

BM25 queries: 500 | Semantic queries: 500


In [2]:
def rrf_run(bm_d, sm_d, qids, k=60, w_bm=1.0, w_sem=1.0, depth=100, out_n=100):
    rows = []
    for qid in qids:
        scores = defaultdict(float)
        for rank, pid in enumerate(bm_d.get(qid, [])[:depth], start=1):
            scores[pid] += w_bm / (k + rank)
        for rank, pid in enumerate(sm_d.get(qid, [])[:depth], start=1):
            scores[pid] += w_sem / (k + rank)
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:out_n]
        for r, (pid, s) in enumerate(ranked, start=1):
            rows.append((qid, pid, r, s))
    return pd.DataFrame(rows, columns=["query_id", "product_id", "rank", "score"])

In [3]:
qids = queries["query_id"].tolist()

hybrid_run = rrf_run(bm_d, sm_d, qids, k=60)

print("BM25:    ", evaluate_run(bm25_run, qrels, k=10))
print("Semantic:", evaluate_run(sem_run, qrels, k=10))
print("Hybrid:  ", evaluate_run(hybrid_run, qrels, k=10))

BM25:     {'Recall@10': 0.3362, 'MRR@10': 0.7332, 'NDCG@10': 0.4952, 'queries_evaluated': 500}
Semantic: {'Recall@10': 0.336, 'MRR@10': 0.7456, 'NDCG@10': 0.4986, 'queries_evaluated': 500}
Hybrid:   {'Recall@10': 0.3621, 'MRR@10': 0.7553, 'NDCG@10': 0.5282, 'queries_evaluated': 500}


In [4]:
rng = np.random.RandomState(42)
shuffled = rng.permutation(qids)
tune_ids = shuffled[:250].tolist()
test_ids = shuffled[250:].tolist()

tune_qrels = qrels[qrels["query_id"].isin(tune_ids)]
test_qrels = qrels[qrels["query_id"].isin(test_ids)]

results = []
for k in [10, 30, 60, 100]:
    for w_sem in [0.5, 1.0, 1.5, 2.0]:
        run = rrf_run(bm_d, sm_d, tune_ids, k=k, w_sem=w_sem)
        s = evaluate_run(run, tune_qrels, k=10)
        results.append({"k": k, "w_sem": w_sem, "NDCG@10": s["NDCG@10"],
                        "MRR@10": s["MRR@10"], "Recall@10": s["Recall@10"]})

grid = pd.DataFrame(results).sort_values("NDCG@10", ascending=False)
grid.head(8)

,k,w_sem,NDCG@10,MRR@10,Recall@10
1,10,1.0,0.5409,0.7760,0.3721
2,10,1.5,0.5407,0.7870,0.3715
6,30,1.5,0.5389,0.7826,0.3691
5,30,1.0,0.5384,0.7780,0.3678
9,60,1.0,0.5356,0.7792,0.3649
10,60,1.5,0.5354,0.7824,0.3631
7,30,2.0,0.5345,0.7830,0.3615
11,60,2.0,0.5344,0.7839,0.3594


In [5]:
best = grid.iloc[0]
best_k, best_w = int(best["k"]), float(best["w_sem"])
print("Best settings:", best_k, best_w)

for name, run_fn in [
    ("BM25", lambda ids: bm25_run[bm25_run["query_id"].isin(ids)]),
    ("Semantic", lambda ids: sem_run[sem_run["query_id"].isin(ids)]),
    ("Hybrid default", lambda ids: rrf_run(bm_d, sm_d, ids, k=60)),
    ("Hybrid tuned", lambda ids: rrf_run(bm_d, sm_d, ids, k=best_k, w_sem=best_w)),
]:
    print(f"{name:15}", evaluate_run(run_fn(test_ids), test_qrels, k=10))

Best settings: 10 1.0
BM25            {'Recall@10': 0.3277, 'MRR@10': 0.7201, 'NDCG@10': 0.4871, 'queries_evaluated': 250}
Semantic        {'Recall@10': 0.3361, 'MRR@10': 0.7377, 'NDCG@10': 0.4966, 'queries_evaluated': 250}
Hybrid default  {'Recall@10': 0.3593, 'MRR@10': 0.7313, 'NDCG@10': 0.5209, 'queries_evaluated': 250}
Hybrid tuned    {'Recall@10': 0.3624, 'MRR@10': 0.7407, 'NDCG@10': 0.5262, 'queries_evaluated': 250}


In [6]:
def minmax_by_query(run):
    run = run.copy()
    g = run.groupby("query_id")["score"]
    mn, mx = g.transform("min"), g.transform("max")
    run["norm"] = (run["score"] - mn) / (mx - mn).replace(0, 1)
    return run

bm_n = minmax_by_query(bm25_run)
sm_n = minmax_by_query(sem_run)

def weighted_run(alpha, qids, out_n=100):
    """alpha = weight of semantic; (1 - alpha) = weight of BM25"""
    bm = bm_n[bm_n["query_id"].isin(qids)].set_index(["query_id", "product_id"])["norm"]
    sm = sm_n[sm_n["query_id"].isin(qids)].set_index(["query_id", "product_id"])["norm"]
    merged = pd.concat([bm.rename("bm"), sm.rename("sm")], axis=1).fillna(0.0)
    merged["score"] = (1 - alpha) * merged["bm"] + alpha * merged["sm"]
    merged = merged.reset_index().sort_values(["query_id", "score"], ascending=[True, False])
    merged["rank"] = merged.groupby("query_id").cumcount() + 1
    return merged[merged["rank"] <= out_n][["query_id", "product_id", "rank", "score"]]

for alpha in [0.3, 0.5, 0.7]:
    run = weighted_run(alpha, tune_ids)
    print(alpha, evaluate_run(run, tune_qrels, k=10))

0.3 {'Recall@10': 0.3685, 'MRR@10': 0.778, 'NDCG@10': 0.5378, 'queries_evaluated': 250}
0.5 {'Recall@10': 0.3765, 'MRR@10': 0.788, 'NDCG@10': 0.5484, 'queries_evaluated': 250}
0.7 {'Recall@10': 0.3623, 'MRR@10': 0.7784, 'NDCG@10': 0.5342, 'queries_evaluated': 250}


In [7]:
final_hybrid = rrf_run(bm_d, sm_d, qids, k=best_k, w_sem=best_w)
final_hybrid.to_parquet(DATA / "run_hybrid.parquet", index=False)

hy_scores = evaluate_run(final_hybrid, qrels, k=10)
print("Hybrid (all 500 queries):", hy_scores)

results_csv = pd.read_csv(DATA / "results.csv")
results_csv = results_csv[results_csv["Method"] != "Hybrid"]
results_csv = pd.concat(
    [results_csv, pd.DataFrame([{"Method": "Hybrid", **hy_scores}])],
    ignore_index=True,
)
results_csv.to_csv(DATA / "results.csv", index=False)
results_csv

Hybrid (all 500 queries): {'Recall@10': 0.3673, 'MRR@10': 0.7583, 'NDCG@10': 0.5335, 'queries_evaluated': 500}


,Method,Recall@10,MRR@10,NDCG@10,queries_evaluated
0,BM25,0.3362,0.7332,0.4952,500
1,Semantic,0.3360,0.7456,0.4986,500
2,Hybrid,0.3673,0.7583,0.5335,500


In [8]:
from src.hybrid import HybridSearch
from src.bm25_search import BM25Search
from src.vector_search import SemanticSearch

corpus = pd.read_parquet(DATA / "corpus.parquet")
emb = np.load(DATA / "corpus_embeddings.npy")

hybrid = HybridSearch(BM25Search(corpus), SemanticSearch(corpus, emb),
                      k=best_k, w_sem=best_w)

for pid, s in hybrid.search("something to keep my coffee hot", 5):
    title = corpus.loc[corpus["product_id"] == pid, "product_title"].iloc[0]
    print(f"{s:.4f}  {title[:90]}")

d:\ML Project - E-commerce product search with semantic ranking\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 781.15it/s]


0.1667  12oz Insulated Coffee Travel Mug Stainless Steel Vacuum Coffee Cup Leakproof with Screw Li
0.1497  Wemordu Smart Coffee Mug Warmer, Coffee Cup Warmer for Desk with Auto Shut Off, Electric B
0.1224  Custom Variety Pack Coffee, Tea, Cappuccino and Hot Chocolate Variety Sampler Pack for Keu
0.1179  12oz Travel Mug, MOMSIV Insulated Coffee Cup with Leakproof Lid, Vacuum Stainless Steel Do
0.1114  [200 Pack] 16 oz. Disposable White Paper Hot Coffee Cups
